In [1]:
import numpy as np
import pandas as pd

# Peak calling

To call peaks in DNase I and ATAC data, we use the developed peak caller hotspot3 [github.com/vierstralab/hotspot3](https://github.com/vierstralab/hotspot3). It is tailored to work on datasets without a control experiment (ATAC-seq and DNase-seq) using adaptive estimation of the background (nonspecific cleavages) with a negative binomial distribution. hotspot3 accounts for variation in both total signal level and signal-to-background ratio along the genome.

In [2]:
!hotspot3 --help

usage: hotspot3 [-h] [--bam BAM] [--chrom_sizes CHROM_SIZES]
                [--fdrs FDRS [FDRS ...]] [--reference REFERENCE] [--cpus CPUS]
                [--outdir OUTDIR] [--debug] [--tempdir TEMPDIR]
                [--mappable_bases MAPPABLE_BASES] [--window WINDOW]
                [--background_window BACKGROUND_WINDOW]
                [--signal_quantile SIGNAL_QUANTILE] [--cutcounts CUTCOUNTS]
                [--signal_parquet SIGNAL_PARQUET]
                [--pvals_parquet PVALS_PARQUET] [--fdrs_parquet FDRS_PARQUET]
                [--chromosomes CHROMOSOMES [CHROMOSOMES ...]] [--save_density]
                id

Call peaks using hotspot3 with local background estimation and MODWT smoothing.

positional arguments:
  id                    Unique identifier of the sample

options:
  -h, --help            show this help message and exit
  --bam BAM             Path to input bam/cram file
  --chrom_sizes CHROM_SIZES
                        Path to chromosome sizes file. If none a

## To obtain hotspot3 peak calls run
```bash
hotspot3 <sample_id> \
    --bam <bam> \
    --chrom_sizes hg38.chrom.sizes \
    --mappable_bases mappable_bases_file.bed.gz \
    --fdrs 0.01 0.05 \
    --cpus 6 \
    --debug 
    
```

### Full README and detailed output files description are available at [github.com/vierstralab/hotspot3](https://github.com/vierstralab/hotspot3)

# Generating sample embeddings from peak calls

Among the produced files there will be peak calls at 1% FDR (`<sample_id>.peaks.fdr0.01.bed.gz`).
To produce the embeddings you need to
1) Overlap peaks with the motif scans
2) Count # of peaks overlapping with each of TF scans
3) Median across TFs belonging to the same non-redundant TF cluster (method described in [Global reference mapping of human transcription factor footprints](https://www.nature.com/articles/s41586-020-2528-x))
4) Z-score normalization

## Step 1: overlap with motif hits
### Note: using remote motif scans results in ~40% slower overlapping. If you need to process multiple samples - consider downloading the scans file locally

In [3]:
!zcat /net/seq/data2/projects/ENCODE4Plus/REGULOME/per_sample//AG95384/peaks/fdr0.01/AG95384.peaks.fdr0.01.bed.gz | wc -l

67012


In [32]:
%%bash

P=/net/seq/data2/projects/ENCODE4Plus/REGULOME/per_sample/AG95384/peaks/fdr0.01/AG95384.peaks.fdr0.01.bed.gz
M=https://resources.altius.org/~sabramov/collaborations/vinson/examples/moods_scans_ref.merged.sorted.bed.gz
#M=/net/seq/data2/projects/sabramov/ENCODE4/dnase-genotypes.v4/round1/output/moods_scans_ref.merged.sorted.bed.gz

time tabix $M -R <(tabix $P chr1) \
    | bedmap --echo-map-id-uniq --delim ';' \
        <(zcat $P) - > AG95384.motif_counts.tsv


real	0m48.680s
user	0m57.490s
sys	0m0.957s


In [73]:
motif_overlap = pd.read_table('AG95384.motif_counts.tsv', names=['motifs'])
n_dhs = len(motif_overlap)
n_dhs

6428

## Step 2: motif counts

In [94]:
motif_counts = motif_overlap['motifs'].str.split(';').explode(
).reset_index().groupby(
    'motifs'
)['index'].agg(
    'nunique'
).reset_index().rename(
    columns={'index': 'n_dhs_with_hit'}
)
motif_counts['motif_id'] = motif_counts['motifs'].str.replace('.pfm', '')
motif_counts['total_dhs'] = n_dhs
motif_counts['motif_proportions'] = motif_counts['n_dhs_with_hit'] / n_dhs
motif_counts.set_index('motif_id', inplace=True)
motif_counts

,motifs,n_dhs_with_hit,total_dhs,motif_proportions
motif_id,,,,
BANP_Grand2021,BANP_Grand2021.pfm,223,6428,0.034692
M00217_2.00,M00217_2.00.pfm,99,6428,0.015401
M00218_2.00,M00218_2.00.pfm,805,6428,0.125233
M00219_2.00,M00219_2.00.pfm,828,6428,0.128811
M00220_2.00,M00220_2.00.pfm,732,6428,0.113877
...,...,...,...,...
MA1984.1,MA1984.1.pfm,485,6428,0.075451
MA1985.1,MA1985.1.pfm,1169,6428,0.181861
MA1986.1,MA1986.1.pfm,1700,6428,0.264468


## Step 3: median within TF cluster

In [79]:
motif_metadata_path = 'https://resources.altius.org/~sabramov/collaborations/vinson/examples/motifs_meta.tsv'

motif_metadata = pd.read_table(motif_metadata_path, index_col=0)
motif_metadata

,cluster,source_id,tf_name,family_name,motif_type,PMID,moods_scans,pfm,motif_hits_path
motif_id,,,,,,,,,
BANP_Grand2021,AC0375,BANP,BANP,Unknown,ChIP-seq,34234345,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00217_2.00,AC0237,ZNF200_H322Y,ZNF200,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00218_2.00,AC0623,ZNF200_REF,ZNF200,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00219_2.00,AC0623,ZNF200_S265Y,ZNF200,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
M00220_2.00,AC0606,SNAI2_D119E,SNAI2,C2H2 ZF,PBM,27013732,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
...,...,...,...,...,...,...,...,...,...
MA1984.1,AC0490,MA1984.1,ZNF667,['C2H2 zinc finger factors'],Unknown,27852650,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
MA1985.1,AC0406,MA1985.1,ZNF669,['C2H2 zinc finger factors'],Unknown,27852650,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...
MA1986.1,AC0124,MA1986.1,ZNF692,['C2H2 zinc finger factors'],Unknown,25690854,/net/seq/data2/projects/sabramov/ENCODE4/dnase...,/home/jvierstra/public_html/projects/motif-clu...,/net/seq/data2/projects/ENCODE4Plus/REGULOME/o...


In [92]:
motif_counts_by_cluster = motif_counts[['n_dhs_with_hit']].join(
    motif_metadata[['cluster', 'tf_name']],
    how='left'
).fillna(0).groupby(
    by="cluster"
)['n_dhs_with_hit'].median().rename('sampleA').to_frame()
motif_counts_by_cluster

,sampleA
cluster,
AC0001,69.0
AC0002,41.5
AC0003,60.5
AC0004,174.0
AC0005,86.5
...,...
AC0633,212.0
AC0634,827.0
AC0635,392.5


## Step 4: z-score normalization

In [95]:
def log_zscore_normalization(x, eta=1e-5):
    x = np.log(x + eta)
    return (x - x.mean()) / x.std()

In [93]:
motif_embedding = (motif_counts_by_cluster / n_dhs).T.apply(
    log_zscore_normalization, eta=1e-5, axis=1
)
motif_embedding

cluster,AC0001,AC0002,AC0003,AC0004,AC0005,AC0006,AC0007,AC0008,AC0009,AC0010,...,AC0628,AC0629,AC0630,AC0631,AC0632,AC0633,AC0634,AC0635,AC0636,AC0637
sampleA,-1.612913,-2.033603,-1.721716,-0.847093,-1.425805,-1.490472,-1.75664,-2.19846,-1.335236,-1.637258,...,0.892459,1.484851,0.809763,-0.986734,-0.50496,-0.683501,0.44403,-0.173325,0.195978,0.675053
